In [5]:
from Bio import SeqIO
import pandas as pd

def get_chain_df(file_csv, chain_type):
   
    file_csv = "../../documentation/clean_entity_mapping.csv"
    chain_type = "../../documentation/pdb_sequences.fasta"
    df = pd.read_csv(file_csv)
    chain_column = "L_chain"

    chains = {
        (str(row['pdb']).lower(), str(row[chain_column]).upper())
        for _, row in df.iterrows()
        if str(row[chain_column]).upper() not in ['NAN', '', 'NONE']
    }

    print(f"how many light chains are in the file: {len(chains)}")
    return chains


def extract_pdb_and_chains(header):
  
    parts = header.split("_")
    pdb_id = parts[0].lower()
    chain_part = next((p for p in parts if p.startswith("chain")), None)
    if chain_part:
        chain_str = chain_part[len("chain"):].split("_")[0]
        chains = [c.upper() for c in chain_str.split(",")]
        return pdb_id, chains
    return pdb_id, []


def extract_unique_chain_sequences(fasta_path, chain_mapping, output_path):
    count_total = 0
    unique_sequences = {}

    for record in SeqIO.parse(fasta_path, "fasta"):
        header = record.id
        sequence = str(record.seq)
        count_total += 1

        pdb_id, chain_ids = extract_pdb_and_chains(header)
        if not chain_ids:
            continue

        if any((pdb_id, chain) in chain_mapping for chain in chain_ids):
            if sequence not in unique_sequences:
                unique_sequences[sequence] = header

    with open(output_path, "w") as out_fasta:
        for seq, header in unique_sequences.items():
            out_fasta.write(f">{header}\n{seq}\n")

    print(f"from {count_total} ")
    print(f"{len(unique_sequences)} unique sequences")

OUTPUT_FASTA = f"{CHAIN_TYPE.lower()}_sequences_deduplicated.fasta"

chain_mapping = get_chain_mapping(CSV_PATH, CHAIN_TYPE)
extract_unique_chain_sequences(FASTA_PATH, chain_mapping, OUTPUT_FASTA)


how many light chains are in the file: 2597
from 4318 
1013 unique sequences
